# 25 - Contract 3: the night's lights, registered and stacked

**Purpose.** Register the 147 good lights of the sky-pair night (`data/session06`) and stack
them, so that the three things notebook `17` had to publish as nulls - *"the frames are on disk
and sufficient, only the engine is missing"* - can be measured:

1. **`eta_comb_registered`** - what a real stack of registered, dithered lights achieves against
   the ideal `sqrt(N)`, at N = 3, 4, 8, 16, laid beside session 03's bias ladder. Rule 5 of
   `protocols/06-sky-pair.md`.
2. **`snr_repeatability`** - how far the SNR estimator moves between two disjoint halves of one
   cell. Rule 6. It is the bar a predicted separation must clear to be a test rather than a tie.
3. **`ranked_pairs`** - the measured column of `sky_pairs.csv`, and MISSION's verdict on the four
   pairs `17` predicted.

This notebook is written for someone *checking* it. `26` reads its files back for someone deciding
what to do next.

**The split approach.** Every light is cut into its four Bayer sub-planes with `SplitCFA` and each
plane is registered on its own (`pjsr/register.js`). Nothing is debayered, so the rule every noise
number here obeys - statistics on the CFA sub-planes - survives registration. **One reference frame
for every cell**, so the signal ROI is the same patch of sky in all four. It is the cell D frame
nearest the middle of the post-flip night, chosen by that rule before anything was measured, and
it is **left out of every stack**: it is the one frame the resampling kernel never touched.

**Noise is measured at 4x4 binning, from differences.** Two images of the same sky on the same
grid - two registered frames, or two stacks of disjoint frames - differ only by noise, so
`stats.diff_sigma` needs no model of the signal. Binning is there because resampling smears each
pixel's noise into its neighbours, which makes a per-pixel spread read low. Binning recovers most
of that but not all (`stats.NOISE_BIN`), so the resampling is measured rather than assumed away:

- **`eta_comb_registered`** = registered single frame against registered stack, in the signal
  ROI. Both have been resampled, so the smear cancels between them and what is left is rejection
  and whatever fails to average away. **This is the number the model uses.**
- **resampling factor**, published beside it as context = registered single frame against *raw*
  single frame, in the same signal
  ROI. The raw pair is lined up by a **whole-pixel** shift (`spatial.integer_offset`), which
  relabels pixels without mixing them, so the raw noise is untouched while the stars and nebula
  still cancel to within half a pixel. How well they cancel is checked, not assumed: raw noise is
  white, so binned 4x4 it must fall to exactly a quarter, and any excess is leftover structure.
  (A first run differenced the raw pair *unaligned*, in the sky corner; at 4x4 the dithered stars
  and nebula outweighed the noise and the factor came out at 0.42, which no resampling kernel can
  do. That method is gone.)
**Why the ideal is the registered frame and not the raw one.** The model's `SNR_sub` describes a
raw sub, so dividing by the resampling factor looks like the principled move. It was tried, and
it came out at 1.00-1.07: better than `sqrt(N)`, which no stack can be. The raw ideal carries
about 10% of extra 4x4 noise - sky that did not cancel within half a pixel, or correlated noise
the raw frames really have - and the two cannot be told apart on this night. That extra inflates
the raw ideal, and with it the efficiency. So resampling is treated as a wash at the scale of
extended signal, which is what the 4x4 factor of about 0.95 says to within that uncertainty, and
the ratio against the raw sub stays in `stack_ladder.csv` as `eta_vs_raw`, labelled as a diagnostic.

Per-pixel numbers are computed beside every binned one, so the size of the smear is on record.

**Fixed before any number was seen**: the reference rule above; the interleave (stack A takes
frames 0, 2, 4, ..., stack B takes 1, 3, 5, ..., so both halves see the same sky drift); the
rejection settings (Winsorized sigma clip 4.0 / 3.0, the arm contract 2 calibrated, plus a
no-rejection arm on the ladder); `additive` normalisation, because *scaling* would rescale the
noise being measured; equal weights; the signal and sky ROIs from `sky_constants.json`, never
moved; and **the repeatability used for a verdict is the larger of two estimates** - the half-split
the protocol names, and the scatter of four quarter-stacks - because a single half-split is one
draw, and a draw that happens to land near zero would turn every tie into a test.

**What it is not for.** Not the per-star clipping count (rule 7): it needs a star detection list,
and it is the star-colour half, not what blocks a ranking. Not the dither-cadence question, which
needs a night shot differently. Not N above 37: this night has 36-37 frames per cell, so the
full-night rungs of hundreds of short subs stay out of reach, and nothing here extrapolates to
them. Not a debayered image of anything.

**It assumes `00_statistics.ipynb`** for why a spread over many thousand pixels is sharp,
`22_pi_arithmetic_read.ipynb` for why averaging in the engine is exact and every loss is the
combination, and `18_sky_pair_read.ipynb` for the night itself.

In [ ]:
import json
import pathlib
import sys
import time

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import fits as F, model as M, pixinsight as PI, spatial, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session06"
# Intermediates are disposable and live on the NAS (CLAUDE.md, D98).  Cells
# below skip work whose result file is already here, so a re-run after a crash
# does not redo an hour of registration; delete the folder to start clean.
TEMP = pathlib.Path(r"Z:\pix\_astro\astropix\temp\contract3\run")
for sub in ("split", "reg", "ref", "stack"):
    (TEMP / sub).mkdir(parents=True, exist_ok=True)

sky = json.loads((RESULTS / "sky_constants.json").read_text())
SIGNAL_ROI = spatial.plane_roi(sky["sky_roi"]["value"]["signal_roi"])
SKY_ROI = spatial.plane_roi(sky["sky_roi"]["value"]["roi"])
T_DEAD = sky["t_dead"]["value"]           # the one the predictions used
K = stats.NOISE_BIN
SCALES = (1, K)                           # per pixel for the record, binned for the number

CELLS = {"A": (50, 30.0), "B": (50, 120.0), "C": (200, 30.0), "D": (200, 120.0)}
RUNGS = [3, 4, 8, 16]                     # 2 is below the engine's floor (NOTES.md 12)
REJECTION = {"none": {"rejection": "none"},
             "winsorized": {"rejection": "winsorized", "sigma_low": 4.0, "sigma_high": 3.0}}
COMMON = {"combination": "average", "normalization": "additive", "weight_mode": "dont_care"}
PLANES = spatial.PLANES

LADDER_CSV = RESULTS / "stack_ladder.csv"
PAIRS_CSV = RESULTS / "stack_pairs.csv"
CONSTANTS = RESULTS / "stack_constants.json"

probe = PI.run(PI.scripts_dir() / "probe.js", {"an_int": 42})
assert probe["ok"], probe.get("error")
core = probe["core"]
print(f"PixInsight {core['version']} build {core['build']}")
print(f"signal ROI {SIGNAL_ROI}, sky ROI {SKY_ROI} (plane pixels), t_dead {T_DEAD:.2f} s")

## 1. The frames, and the reference

The good frames are exactly `17`'s: after the meridian flip, and not rejected by its cloud test.
Nothing is re-selected here. Within a cell they are ordered by time, which is what the interleave
below relies on.

In [ ]:
frames = pd.read_csv(RESULTS / "sky_frames.csv", parse_dates=["t"])
good = frames[~frames.flipped & ~frames.rejected].sort_values("t").reset_index(drop=True)
good["stem"] = [f"{c}_{t:%H%M%S}" for c, t in zip(good.cell, good.t)]
assert good.stem.is_unique

# The reference: cell D, nearest the middle of the post-flip night.  Chosen by
# rule, not by look; D because 120 s at gain 200 has the most stars per plane.
mid = good.t.min() + (good.t.max() - good.t.min()) / 2
d = good[good.cell == "D"]
REF = d.loc[(d.t - mid).abs().idxmin()]
print(f"reference: {REF.file}  ({REF.t:%H:%M:%S}, night middle {mid:%H:%M:%S})")

stackable = good[good.stem != REF.stem]
print(stackable.groupby("cell").size().rename("frames").to_frame().T.to_string())

## 2. Registration, one launch per cell

About twelve minutes per cell: splitting is a few seconds a frame, and StarAlignment is about four
seconds a frame per plane. Each launch writes its result to `TEMP`, and a result already there is
reused. The reference is split in every launch - it is the same file, so the same planes.

In [ ]:
reg = {}
for cell in CELLS:
    out = TEMP / f"register_{cell}.json"
    if not out.exists():
        rows = stackable[stackable.cell == cell]
        job = {"reference": PI.pi_path(DATA / REF.file), "outdir": PI.pi_path(TEMP),
               "interpolation": "lanczos3",
               "frames": [{"path": PI.pi_path(DATA / f), "stem": s}
                          for f, s in zip(rows.file, rows.stem)]}
        t0 = time.time()
        r = PI.run(PI.scripts_dir() / "register.js", job, timeout=3600)
        assert r["ok"], r.get("error")
        r.pop("log", None)
        out.write_text(json.dumps(r, indent=1))
        print(f"cell {cell}: {len(rows)} frames registered in {time.time() - t0:.0f} s")
    reg[cell] = json.loads(out.read_text())
print("settings:", reg["A"]["settings"])

In [ ]:
# One row per frame and plane.  PI's names: G2 and G1 swapped against a naive
# RGGB reading was already undone inside register.js (NOTES.md 6).
qual = pd.DataFrame([
    {"cell": c, "stem": f["stem"], "plane": p, "ok": v["ok"], "path": v.get("output", ""),
     "matches": v.get("matches"), "rms_error": v.get("rms_error"),
     "dx": v.get("H13"), "dy": v.get("H23")}
    for c, r in reg.items() for f in r["frames"] for p, v in f["planes"].items()])
print(f"{(~qual.ok).sum()} of {len(qual)} frame-planes failed to register")
print(qual.groupby(["cell", "plane"]).agg(matches=("matches", "min"),
                                          rms_worst=("rms_error", "max")).unstack().round(3).to_string())
assert qual.ok.all(), "a frame that did not register cannot be stacked; see the failures above"
print(f"\ndither + drift against the reference: |dx| up to {qual.dx.abs().max():.1f}, "
      f"|dy| up to {qual.dy.abs().max():.1f} plane pixels")

## 3. Single frames: registered against raw

Consecutive frames of a cell are paired - (0, 1), (2, 3), ... - which is the same partition the
stacks use. Each pair gives two noises at two scales, both in the signal ROI:

- **registered**: the ideal the combination efficiency is measured against.
- **raw, lined up by a whole-pixel shift**: found by phase correlation on the whole plane, applied
  by moving the box in the second frame. Registered over raw is the resampling factor.

`raw_white` is the raw 4x4 noise over the raw per-pixel noise. White noise gives exactly 0.25;
anything above it is sky that did not cancel, and it is carried into the published note.

In [ ]:
def plane_of_raw(file, plane):
    return stats.to_adc(spatial.split(F.read(DATA / file)[0])[plane]).astype(np.float64)


path_of = qual.set_index(["stem", "plane"]).path
singles = []
for cell in CELLS:
    rows = stackable[stackable.cell == cell].reset_index(drop=True)
    for i in range(0, len(rows) - 1, 2):
        a, b = rows.iloc[i], rows.iloc[i + 1]
        for p in PLANES:
            ra = PI.read_adc(path_of[a.stem, p])
            rb = PI.read_adc(path_of[b.stem, p])
            wa, wb = plane_of_raw(a.file, p), plane_of_raw(b.file, p)
            assert spatial.cut(ra, SIGNAL_ROI).min() > 0 and spatial.cut(rb, SIGNAL_ROI).min() > 0
            dx, dy = spatial.integer_offset(wa, wb)
            x, y, w, h = SIGNAL_ROI
            box_a, box_b = spatial.cut(wa, SIGNAL_ROI), spatial.cut(wb, (x + dx, y + dy, w, h))
            row = {"cell": cell, "plane": p, "pair": i // 2, "dx": dx, "dy": dy}
            for k in SCALES:
                row[f"reg_signal_{k}"] = stats.diff_sigma(spatial.cut(ra, SIGNAL_ROI),
                                                          spatial.cut(rb, SIGNAL_ROI), k)
                row[f"raw_signal_{k}"] = stats.diff_sigma(box_a, box_b, k)
            singles.append(row)
singles = pd.DataFrame(singles)
print(f"{len(singles)} pair-planes; raw shifts up to |dx| {singles.dx.abs().max()}, "
      f"|dy| {singles.dy.abs().max()} plane pixels")

one = singles.groupby(["cell", "plane"]).mean(numeric_only=True)
for k in SCALES:
    one[f"resample_{k}"] = one[f"reg_signal_{k}"] / one[f"raw_signal_{k}"]
one["raw_white"] = one[f"raw_signal_{K}"] / one["raw_signal_1"]
print(one[[f"reg_signal_{k}" for k in SCALES] + [f"resample_{k}" for k in SCALES] + ["raw_white"]]
      .round(4).to_string())
print(f"\nraw_white {one.raw_white.mean():.4f} (range {one.raw_white.min():.4f} to "
      f"{one.raw_white.max():.4f}); white noise gives {1 / K:.4f}")

## 4. The stacks

Per cell and plane, all from one launch per cell:

| label | frames | arms |
|---|---|---|
| `A{N}`, `B{N}` for N in 3, 4, 8, 16 | the first N of the even / odd frames | none, winsorized |
| `hA`, `hB` | all the even / all the odd frames, trimmed to equal length | none, winsorized |
| `q0` ... `q3` | frames `i::4`, trimmed to equal length | winsorized |
| `full` | every stackable frame | winsorized |

`hA`/`hB` are the top rung of the ladder as well as the half-split. `q0` and `q2` are drawn from
the even frames and `q1` and `q3` from the odd ones, so each half has two quarters of its own.

In [ ]:
def plan(rows):
    ev, od = list(rows.stem[0::2]), list(rows.stem[1::2])
    m, m4 = min(len(ev), len(od)), len(rows) // 4
    sets = {f"A{n}": ev[:n] for n in RUNGS} | {f"B{n}": od[:n] for n in RUNGS}
    sets |= {"hA": ev[:m], "hB": od[:m]}
    quarters = {f"q{i}": list(rows.stem[i::4])[:m4] for i in range(4)}
    return sets, quarters, {"full": list(rows.stem)}


stacks = {}
for cell in CELLS:
    out = TEMP / f"integrate_{cell}.json"
    rows = stackable[stackable.cell == cell].reset_index(drop=True)
    sets, quarters, full = plan(rows)
    runs = []
    for p in PLANES:
        for arm, groups in (("none", sets), ("winsorized", sets | quarters | full)):
            for label, stems in groups.items():
                f = TEMP / "stack" / f"{cell}_{p}_{arm}_{label}.fits"
                runs.append({"label": f"{cell}|{p}|{arm}|{label}", "output": PI.pi_path(f),
                             "frames": [PI.pi_path(path_of[s, p]) for s in stems],
                             **COMMON, **REJECTION[arm]})
    if not out.exists():
        t0 = time.time()
        r = PI.run(PI.scripts_dir() / "integrate.js", {"runs": runs}, timeout=5400)
        assert r["ok"], r.get("error")
        r.pop("log", None)
        out.write_text(json.dumps(r, indent=1))
        print(f"cell {cell}: {len(runs)} integrations in {time.time() - t0:.0f} s")
    r = json.loads(out.read_text())
    bad = [x["label"] for x in r["runs"] if not x["ok"]]
    assert not bad, f"integrations that failed: {bad}"
    for x in r["runs"]:
        stacks[x["label"]] = {"n": x["n"], "path": x["output"], "settings": x["settings"]}
print(f"{len(stacks)} stacks; settings read back, e.g.:",
      next(iter(stacks.values()))["settings"])

## 5. The ladder: combination efficiency, and `eta_comb_registered`

`sigma_N` is the noise of one stack of N, from the difference of its A and B twins. The ideal is
the mean registered single-frame noise of that cell and plane - a mean over all its pairs, not one
frame that is also inside the stack, which is the flaw `22` section 7 named in contract 2's ladder.

In [ ]:
def load(cell, p, arm, label):
    return PI.read_adc(stacks[f"{cell}|{p}|{arm}|{label}"]["path"])


ladder = []
for cell in CELLS:
    for p in PLANES:
        for arm in REJECTION:
            for a_lab, b_lab in [(f"A{n}", f"B{n}") for n in RUNGS] + [("hA", "hB")]:
                n = stacks[f"{cell}|{p}|{arm}|{a_lab}"]["n"]
                sa = spatial.cut(load(cell, p, arm, a_lab), SIGNAL_ROI)
                sb = spatial.cut(load(cell, p, arm, b_lab), SIGNAL_ROI)
                row = {"cell": cell, "gain": CELLS[cell][0], "exptime": CELLS[cell][1],
                       "plane": p, "arm": arm, "n": n}
                for k in SCALES:
                    sig = stats.diff_sigma(sa, sb, k)
                    comb = PI.eta_comb(one.loc[(cell, p), f"reg_signal_{k}"], sig, n)
                    row[f"sigma_{k}"] = sig
                    row[f"eta_comb_{k}"] = comb
                    # diagnostic only -- see the purpose cell for why it is not published
                    row[f"eta_vs_raw_{k}"] = comb / one.loc[(cell, p), f"resample_{k}"]
                ladder.append(row)
ladder = pd.DataFrame(ladder)
view = ladder.pivot_table(index=["arm", "n"], columns="cell",
                          values=f"eta_comb_{K}", aggfunc="mean")
print(f"eta_comb_registered at {K}x{K}, mean over the four planes:")
print(view.round(4).to_string())

In [ ]:
dark = json.loads((RESULTS / "dark_constants.json").read_text())["eta_comb"]["value"]
w = ladder[ladder.arm == "winsorized"]
by_n = w.groupby("n")[[f"eta_comb_{K}", "eta_comb_1", f"eta_vs_raw_{K}"]].agg(["mean", "std"])
print("winsorized, over cells and planes -- beside session 03's bias ladder (no registration):")
for n, r in by_n.iterrows():
    ref = dark.get(str(n))
    print(f"  N={n:3d}  eta_comb_registered {r[(f'eta_comb_{K}', 'mean')]:.4f} "
          f"+- {r[(f'eta_comb_{K}', 'std')]:.4f}   per-pixel {r[('eta_comb_1', 'mean')]:.4f}"
          f"   [vs raw sub {r[(f'eta_vs_raw_{K}', 'mean')]:.4f}]"
          f"   bias ladder {ref if ref is not None else '  -   '}")
res = one[[f"resample_{k}" for k in SCALES]]
print(f"\nresampling factor, registered / raw single-frame noise, signal ROI:")
print(f"  per pixel {res['resample_1'].mean():.4f} +- {res['resample_1'].std():.4f}   "
      f"at {K}x{K} {res[f'resample_{K}'].mean():.4f} +- {res[f'resample_{K}'].std():.4f}")

## 6. SNR per cell, and the repeatability of the estimator

**Signal** is `stats.extended_signal`: the median of the signal ROI above the median of the sky
ROI, on the full stack. The sky ROI sits 32 plane pixels from the corner, so the dither carries
some frames off it; their zero-filled border is rejected by `ImageIntegration`'s low range clip
(on by default, read back in the settings) and the stack there is built from the frames that
cover it. **Noise** is the full stack's, from the half-split scaled by the frame
count, at 4x4.

**Repeatability**, two ways, fixed in advance:
- *half-split* (the protocol's rule 6): each half gets its own SNR - its signal from itself, its
  noise from its own two quarters - and the relative difference between them, halved, is the
  relative error of the full-stack SNR (two draws of an N/2 stack differ by sqrt(2) of one draw, and
  the full stack is sqrt(2) better than one of them).
- *quarter scatter*: the SNR of each of the four quarters, with the quarter noise from `q0`/`q2`
  and `q1`/`q3`; their relative standard deviation, halved, is the same quantity with three degrees
  of freedom instead of one.

The larger is used. A pair's repeatability combines its two cells in quadrature.

In [ ]:
def snr_of(cell, p, label, noise):
    s = load(cell, p, "winsorized", label)
    return stats.extended_signal(spatial.cut(s, SIGNAL_ROI), spatial.cut(s, SKY_ROI)) / noise


def sig_of(cell, p, a, b):
    return stats.diff_sigma(spatial.cut(load(cell, p, "winsorized", a), SIGNAL_ROI),
                            spatial.cut(load(cell, p, "winsorized", b), SIGNAL_ROI), K)


snr = []
for cell in CELLS:
    n_full = stacks[f"{cell}|R|winsorized|full"]["n"]
    m = stacks[f"{cell}|R|winsorized|hA"]["n"]
    m4 = stacks[f"{cell}|R|winsorized|q0"]["n"]
    for p in PLANES:
        s_half = sig_of(cell, p, "hA", "hB")
        s_qa, s_qb = sig_of(cell, p, "q0", "q2"), sig_of(cell, p, "q1", "q3")
        full = snr_of(cell, p, "full", s_half * np.sqrt(m / n_full))
        ha = snr_of(cell, p, "hA", s_qa * np.sqrt(m4 / m))
        hb = snr_of(cell, p, "hB", s_qb * np.sqrt(m4 / m))
        q = [snr_of(cell, p, f"q{i}", (s_qa, s_qb, s_qa, s_qb)[i]) for i in range(4)]
        snr.append({"cell": cell, "plane": p, "n": n_full, "snr": full,
                    "rep_half_pct": 100 * abs(ha / hb - 1) / 2,
                    "rep_quarter_pct": 100 * np.std(q, ddof=1) / np.mean(q) / 2})
snr = pd.DataFrame(snr)
snr["rep_pct"] = snr[["rep_half_pct", "rep_quarter_pct"]].max(axis=1)
print(snr.round(4).to_string(index=False))

## 7. The four pairs

Green is the mean of the G1 and G2 SNRs - the two greens see the same sky through the same filter,
and `17` predicted in green. Each cell's SNR is put on the footing of the prediction with
`model.snr_per_root_night`: divided by the root of the wall clock its stack cost, at the same
`t_dead` the prediction used. `model.verdict` applies MISSION's rule.

In [ ]:
def cell_snr(cell, plane):
    rows = snr[snr.cell == cell].set_index("plane")
    planes = ["G1", "G2"] if plane == "green" else [plane]
    s = rows.loc[planes, "snr"].mean()
    rep = float(np.sqrt((rows.loc[planes, "rep_pct"] ** 2).mean()))
    return M.snr_per_root_night(s, int(rows.n.iloc[0]), CELLS[cell][1], T_DEAD), rep


pred = pd.read_csv(RESULTS / "sky_pairs.csv")
out = []
for _, r in pred.iterrows():
    first, second = r.pair.split("->")
    s1, rep1 = cell_snr(first, r.plane)
    s2, rep2 = cell_snr(second, r.plane)
    measured = 100 * (s2 / s1 - 1)
    rep = float(np.hypot(rep1, rep2))
    out.append({"pair": r.pair, "why": r.why, "plane": r.plane, "t_dead_s": T_DEAD,
                "predicted_pct": r.predicted_pct, "measured_pct": measured,
                "repeatability_pct": rep,
                "verdict": M.verdict(r.predicted_pct, measured, rep)})
pairs = pd.DataFrame(out)
print(pairs.round(3).to_string(index=False))

done = pairs[pairs.plane == "green"]
passed = done[done.verdict == "pass"]
straddle = passed.why.str.contains("straddles").any()
print(f"\ngreen: {len(passed)} of {len(done)} pairs pass, "
      f"{(done.verdict == 'tie').sum()} are ties; a passing pair straddles HCG: {straddle}")
print("MISSION's definition of done (three passes, one straddling HCG):",
      "MET" if len(passed) >= 3 and straddle else "NOT MET")

## 8. Publish

`stack_ladder.csv` holds every rung, both arms and both scales; `stack_pairs.csv` the four pairs in
both planes; `stack_constants.json` the constants, each with its provenance. The eta tables
published as constants are the **winsorized arm at 4x4**, averaged over cells and planes, because a
model table is one curve; the per-cell, per-plane spread is their uncertainty and the rows behind
it are in the CSV.

In [ ]:
measured_on = str(good.t.min().date())
n_src = int(len(stackable))


def constant(value, unit, uncertainty, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty,
            "source_frames": n_src, "measured_on": measured_on,
            "notebook": "25_contract3_stacks.ipynb", "note": note}


def r6(v):
    return None if v is None or not np.isfinite(v) else round(float(v), 6)


def table(arm, col):
    g = ladder[ladder.arm == arm].groupby("n")[col]
    return {str(n): r6(v) for n, v in g.mean().items()}, {str(n): r6(v) for n, v in g.std().items()}


SETTINGS = (f"Winsorized sigma clip {REJECTION['winsorized']['sigma_low']}/"
            f"{REJECTION['winsorized']['sigma_high']}, average, additive normalisation, equal "
            f"weights, Lanczos-3 registration of each Bayer plane separately (SplitCFA, no "
            f"debayer), dither every frame, {K}x{K} binning, signal ROI {SIGNAL_ROI} in plane pixels")
v_w, u_w = table("winsorized", f"eta_comb_{K}")
v_n, u_n = table("none", f"eta_comb_{K}")
excess = 100 * (K * one.raw_white.mean() - 1)

constants = {
    "eta_comb_registered": constant(
        v_w, "measured sd reduction against the ideal sqrt(N) of a registered sub", u_w,
        "rule 5, keyed by stack size N.  " + SETTINGS + ".  The ideal is the registered single "
        "sub, so resampling cancels and this is the combination: rejection and whatever fails to "
        "average away.  Resampling is treated as a wash at 4x4 -- see resampling_factor.  "
        "Uncertainty is the spread over 4 cells x 4 planes.  N stops at the half-stack of this "
        "night; nothing here says what happens at N of hundreds"),
    "eta_comb_registered_no_rejection": constant(
        v_n, "measured sd reduction against the ideal sqrt(N) of a registered sub", u_n,
        "the same ladder with rejection off, so the gap to eta_comb_registered is what "
        "Winsorized clipping costs or buys on real registered lights"),
    "resampling_factor": constant(
        {k: r6(one[f"resample_{k}"].mean()) for k in SCALES},
        "registered / raw single-frame noise, keyed by bin size",
        {k: r6(one[f"resample_{k}"].std()) for k in SCALES},
        "context, not a correction: nothing divides by it.  Measured in the signal ROI, "
        "registered pairs against raw pairs lined up by a whole-pixel shift, which moves no noise "
        "between pixels.  Below 1 means registration smoothed the noise; the per-pixel figure is "
        "how far a per-pixel spread would have been fooled.  The raw 4x4 / 1x1 ratio was "
        f"{one.raw_white.mean():.4f} against 0.25 for white noise, so the raw binned noise "
        f"carries about {excess:.1f}% extra -- sky that did not cancel within half a pixel, or "
        "correlated noise the raw frames really have, which this night cannot separate.  That "
        "extra inflates the raw side, so the binned factor reads LOW by up to that much, and "
        "dividing the efficiency by it would read HIGH: tried, it gave 1.00-1.07, above sqrt(N).  "
        "That is why eta_comb_registered uses the registered sub as its ideal"),
    "snr_repeatability": constant(
        {c: r6(snr[snr.cell == c].rep_pct.mean()) for c in CELLS},
        "% relative error of a cell's full-stack SNR, mean over planes", None,
        "rule 6.  The larger of the half-split and the quarter scatter, chosen in advance because "
        "a half-split is one draw.  Per plane in stack_pairs.csv via each pair's repeatability"),
    "ranked_pairs": constant(
        {f"{r.pair} {r.plane}": r.verdict for r in pairs.itertuples()},
        "verdict per pair and plane: pass, fail or tie", None,
        "MISSION's definition of done, applied by model.verdict: a tie when the predicted "
        "separation is inside the repeatability; a pass when the winner is right and the "
        "measured ratio is within 10% of the predicted one.  Predictions are 17's, unchanged"),
}
with open(CONSTANTS, "w") as f:
    json.dump(constants, f, indent=2)
ladder.round(6).to_csv(RESULTS / "stack_ladder.csv", index=False)
pairs.round(6).to_csv(RESULTS / "stack_pairs.csv", index=False)
M.Constants.load(CONSTANTS)       # the provenance gate, before anything else reads it
print("wrote", CONSTANTS.name, LADDER_CSV.name, PAIRS_CSV.name)

## 9. Clean up

The registered planes and stacks are disposable (D98) and several GB. They are removed only once
the three files above exist.

In [ ]:
import shutil
assert CONSTANTS.exists() and LADDER_CSV.exists() and PAIRS_CSV.exists()
shutil.rmtree(TEMP, ignore_errors=True)
print("removed", TEMP)